# Wildfire event mapping using Sentinel-2 data

Map the extent of wildfires and classify their severity using Sentinel-2 data.

Generate a pre event mosaic and use the first available post event image. 
Calculate the NBR for both and generate a change image using the difference between pre and post event and apply thresholds to classify the severity of the areas affected by the fire. 

In [19]:
import openeo
import rasterio
from openeo.processes import ProcessBuilder
from openeo.processes import quantiles
from folium.plugins import Draw
import leafmap
from IPython.display import JSON
from shapely.geometry import shape
from openeo.processes import quantiles, array_element, median
import rioxarray as rxr

In [ ]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/fire/Fontainebleau")
#out_dir.mkdir()
event = "Fontainebleau_2026"

## 1) Define time frame and extent

In [3]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [4]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 2.381287, 'south': 48.298726, 'east': 2.702637, 'north': 48.439223}


In [ ]:
# dates stelvio fire
PRE_DATE  = ("2024-07-01", "2024-07-31")   
POST_DATE = ("2025-06-15", "2025-06-20")   

In [ ]:
# dates tuscany fire
PRE_DATE  = ("2025-05-01", "2025-05-30")   
POST_DATE = ("2026-05-27", "2026-05-30")   

In [3]:
# dates fontainebleau fire
PRE_DATE  = ("2025-07-01", "2025-07-30")   
POST_DATE = ("2026-07-20", "2026-07-25")   

### 2) Authentificate and load the pre and post Sentinel-2 cube

In [4]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
connection.authenticate_oidc()

Authenticated using refresh token.
Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [ ]:
BANDS   = ["B08", "B12"]          
BAD_SCL = [1, 3, 7, 8, 9, 10]

TIMESPAN = {
    "PRE": PRE_DATE,
    "POST": POST_DATE,
}

def load_s2(temporal_extent):
    bands = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=20,
    ).resample_spatial(resolution=10, method="bilinear")

    scl = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=["SCL"],
        max_cloud_cover=20,
    ).resample_spatial(resolution=10, method="near")   # categorical -> nearest

    return bands, scl


def cloud_mask(scl):
    s = scl.band("SCL")
    m = s == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (s == v)
    return m


def masked_cube(temporal_extent):
    bands, scl = load_s2(temporal_extent)
    return bands.mask(cloud_mask(scl))


def composite_pre(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: array_element(quantiles(data=x, probabilities=[0.25]), index=0),
    )


def composite_post(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: median(x),
    )


def nbr(mosaic):
    """NBR = (NIR - SWIR2) / (NIR + SWIR2) = (B08 - B12) / (B08 + B12)"""
    def _nbr(b):
        b08 = b[0]   
        b12 = b[1]
        return (b08 - b12) / (b08 + b12)
    return mosaic.reduce_dimension(dimension="bands", reducer=_nbr)


COMPOSITES = {"PRE": composite_pre, "POST": composite_post}

jobs = {}
for name, extent in TIMESPAN.items():
    print(f"Starting {name}...")
    result = nbr(COMPOSITES[name](extent)).save_result(format="GTiff")
    job = result.create_job(title=f"S2_NBR_{name}")
    job.start()
    jobs[name] = job
    print(f"{name} started: {job.job_id}")

Starting PRE...
PRE started: j-2610051405494b388775b8d52814f802
Starting POST...
POST started: j-2610051405534c7789ea738161086e64


In [ ]:
def latest_finished(jobs, title):
    matches = [j for j in jobs if j.get("title") == title and j["status"] == "finished"]
    if not matches:
        raise ValueError(f"No finished job titled {title}")
    return max(matches, key=lambda j: j["created"])

jobs = connection.list_jobs()
pre_job  = latest_finished(jobs, "S2_NBR_PRE")
post_job = latest_finished(jobs, "S2_NBR_POST")

In [18]:
file_pre_nbr = out_dir/ f"pre_{event}_NBR.tiff"
file_post_nbr = out_dir/ f"post_{event}_NBR.tiff"

connection.job(pre_job["id"]).get_results().download_file(file_pre_nbr)
connection.job(post_job["id"]).get_results().download_file(file_post_nbr)

PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/fire/Fontainebleau/post_Fontainebleau_2026_NBR.tiff')

In [22]:
pre  = rxr.open_rasterio(file_pre_nbr,  masked=True)
post = rxr.open_rasterio(file_post_nbr, masked=True)

dnbr = pre - post   # or post - pre
dnbr.rio.to_raster(out_dir / f"dnbr_{event}.tiff")